In [1]:
# %% [markdown]
# # RMSE and Hellinger Distance Computation (Stage 5 Pipeline)
#
# Takes the LNLM parameters fitted in Notebook 1 (Stage 5 data source)
# and:
#
# 1. Evaluates each factor's model on the estimation window
# 2. Computes the RMSE over multiple evaluation windows
# 3. Applies the 80% noise filter
# 4. Computes the monthly Hellinger distance and SRI (aggregate/combined)
# 5. Tests significance with both monthly and quarterly sampling
# 6. Checks robustness to filter threshold
# 7. Saves everything Notebook 3 needs -- including RMSE matrices AND a
#    shared per-month, PER-WINDOW imputation array, so the theme-level
#    H^2 identity in Notebook 3 holds EXACTLY, not just approximately.
#
# ═══════════════════════════════════════════════════════════════════════
# WHY THE IMPUTATION FIX MATTERS (and why it's now per-window)
# ═══════════════════════════════════════════════════════════════════════
#
# The old notebook's compute_hellinger() imputed a missing factor's RMSE
# for a given month using np.nanmedian() computed over WHATEVER SUBSET
# of factors was passed in -- the full surviving set for the aggregate
# H, but a single theme's factors for a per-theme H. A factor with a
# missing month got a DIFFERENT imputed value depending on which grouping
# it was summed within, which breaks
#     H_aggregate^2 = sum_over_themes( H_theme^2 )
#
# FIX (v1, corrected here to v2): compute ONE fill value per (month,
# surviving factor), from the FULL surviving set, ONCE -- but do this
# SEPARATELY for each evaluation window (1-day, 5-day, 10-day), not just
# once from the 5-day RMSE and reused everywhere. A NaN in rmse[1] (the
# 1-day window) represents a missing 1-day RMSE; filling it with a value
# drawn from the 5-day RMSE distribution for that month is not just
# "a different grouping's median" (which the v1 fix already handled) --
# it is a different, unrelated quantity with a different scale. The H^2
# identity would still technically hold under the v1 (single-window)
# fix, since the SAME wrong value would be reused consistently across
# any aggregate-vs-theme comparison at a given window -- but the
# substituted number itself would misrepresent what "typical RMSE at
# this window, this month" means for the 1-day and 10-day windows
# specifically.
#
# A diagnostic below (Part B.1) checks whether this actually matters in
# practice for this dataset: given the Notebook 1 pre-flight already
# confirmed zero NaNs in the raw feature panel and target, a NaN can only
# appear in rmse[w][m,f] for a SURVIVING factor via an occasional
# single-month std<1e-10 or valid.sum()<100 fallback inside the fitting
# loop -- plausibly rare or absent. If the diagnostic count is 0, the
# per-window fix is a no-op in practice; it is still implemented properly
# below (not skipped) so this notebook is correct regardless of what any
# future dataset's diagnostic shows.
#
# ═══════════════════════════════════════════════════════════════════════
DATASET_NAME = "agg_full_moments"   # or "agg_means"
# ═══════════════════════════════════════════════════════════════════════

# ==========================================================================
# %% [markdown]
# ## Part A: Setup and Data Loading

# %%
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import norm as normal_dist
from pathlib import Path
import time

ROOT = Path("../..")
RESULTS_ROOT = ROOT / "Data" / "Results" / "Reproducing_Hellinger_Polymodel"
DATA_PM = RESULTS_ROOT / "intermediate"

print(f"DATASET_NAME: {DATASET_NAME}")
print(f"Reading/writing from: {DATA_PM}")

# %%
# ── A.1: Load LNLM parameters from Notebook 1 ──

params_path = DATA_PM / f"lnlm_params_{DATASET_NAME}.npz"
assert params_path.exists(), (
    f"Not found: {params_path} -- run Notebook 1 with DATASET_NAME="
    f"'{DATASET_NAME}' first."
)
params = np.load(params_path, allow_pickle=True)

beta_lin    = params["beta_lin"]
beta_nonlin = params["beta_nonlin"]
mu_star     = params["mu_star"]
y_mean      = params["y_mean"]
param_dates = params["dates"]
feature_names = list(params["feature_names"])

n_months = len(param_dates)
n_features = len(feature_names)

print(f"Parameters loaded: {n_months} months x {n_features} features")

# %%
# ── A.2: Load features (unlagged) and target ──

features_path = DATA_PM / f"polymodel_features_unlagged_{DATASET_NAME}.parquet"
target_path   = DATA_PM / f"polymodel_target_{DATASET_NAME}.parquet"

features_df = pd.read_parquet(features_path)
target_df   = pd.read_parquet(target_path)

all_dates      = pd.to_datetime(features_df["date"]).values
feature_cols   = [c for c in features_df.columns if c != "date"]
features_raw   = features_df[feature_cols].values.astype(np.float64)
monthly_target = target_df["monthly_return"].values.astype(np.float64)
daily_returns  = target_df["daily_return"].values.astype(np.float64)

assert feature_cols == feature_names, (
    "Feature columns in the unlagged parquet do not match feature_names "
    "in the parameter file -- these must come from the SAME Notebook 1 "
    "run for the same DATASET_NAME. STOP."
)
print(f"  ✓ Feature column alignment confirmed ({len(feature_cols)} features)")

print(f"Features: {features_raw.shape}")
print(f"Target:   {len(monthly_target)} rows")

# %%
# ── A.3: Apply the 21-day lag ──

LAG = 21
feature_values = np.full_like(features_raw, np.nan)
feature_values[LAG:] = features_raw[:-LAG]

print(f"Features lagged by {LAG} days. First valid row: {LAG}")

# %%
# ── A.4: Load metadata ──

meta_path = DATA_PM / f"polymodel_feature_meta_{DATASET_NAME}.csv"
meta = pd.read_csv(meta_path, dtype={"subtheme_id": str, "theme_id": str})

has_valid_params = np.any(~np.isnan(mu_star), axis=0)
combined_feature_idx = np.where(has_valid_params)[0]

print(f"\nCombined dataset: {len(combined_feature_idx)} features with valid parameters")
print(f"  (out of {n_features} total, {n_features - len(combined_feature_idx)} cleaned out)")

# %%
# ── A.5: Define evaluation windows and month-end row indices ──

WINDOW = 1260

EVAL_WINDOWS = [10, 5, 1]
WINDOW_NAMES = {
    10:   "2 weeks",
    5:    "1 week",
    1:    "1 day",
}

month_rows = np.array([
    int(np.searchsorted(all_dates, d)) for d in param_dates
])

print(f"\nEvaluation windows: {[WINDOW_NAMES[w] for w in EVAL_WINDOWS]}")
print(f"Estimation dates: {n_months} months "
      f"({pd.Timestamp(param_dates[0]).strftime('%Y-%m')} to "
      f"{pd.Timestamp(param_dates[-1]).strftime('%Y-%m')})")

# %%
# ── A.6: Logistic regression helper ──

def logistic_t(signal, fwd_return, label=""):
    valid = ~np.isnan(signal) & ~np.isnan(fwd_return)
    if valid.sum() < 30:
        return None
    y = (fwd_return[valid] > 0).astype(float)
    X = sm.add_constant(signal[valid])
    try:
        fit = sm.Logit(y, X).fit(disp=0)
        return {"label": label, "n": int(valid.sum()),
                "t": fit.tvalues[1], "p": fit.pvalues[1]}
    except Exception:
        return None


# ==========================================================================
# %% [markdown]
# ## Part B: Compute RMSE Across All Evaluation Windows

# %%
rmse = {}
for w in EVAL_WINDOWS:
    rmse[w] = np.full((n_months, n_features), np.nan)

print(f"Computing RMSE: {n_features} features x {n_months} months "
      f"x {len(EVAL_WINDOWS)} windows...")
t0 = time.perf_counter()

for m in range(n_months):
    idx = month_rows[m]
    ws = idx - WINDOW + 1
    we = idx + 1

    y = monthly_target[ws:we]
    X = feature_values[ws:we]
    ym = y_mean[m]

    bl = beta_lin[m]
    bn = beta_nonlin[m]
    ms = mu_star[m]

    X2 = X * X
    nonlin_pred = (X * bn[:, 0] +
                   (X2 - 1) * bn[:, 1] +
                   (X2 * X - 3 * X) * bn[:, 2] +
                   (X2 * X2 - 6 * X2 + 3) * bn[:, 3])
    lin_pred = X * bl
    y_hat = ym + ms * nonlin_pred + (1 - ms) * lin_pred

    sq_resid = (y[:, np.newaxis] - y_hat) ** 2

    for w in EVAL_WINDOWS:
        rmse[w][m] = np.sqrt(np.nanmean(sq_resid[-w:], axis=0))

    if (m + 1) % 20 == 0:
        elapsed = time.perf_counter() - t0
        eta = elapsed / (m + 1) * (n_months - m - 1)
        print(f"  Month {m+1}/{n_months}  [{elapsed:.0f}s elapsed, ~{eta:.0f}s remaining]")

print(f"  Done in {time.perf_counter() - t0:.1f}s")

print(f"\n  Mean RMSE by evaluation window:")
for w in EVAL_WINDOWS:
    print(f"    {WINDOW_NAMES[w]:12s}: {np.nanmean(rmse[w]):.6f}")

# %%
# ── B.1: DIAGNOSTIC -- does the imputation mismatch actually matter here?
#
# Given Notebook 1's pre-flight already confirmed zero NaNs in the raw
# feature panel and target, a NaN in rmse[w][m,f] for a SURVIVING factor
# (one that already passed the >=50%-valid-months cleanup criterion) can
# only arise from an occasional single-month std<1e-10 or valid.sum()<100
# fallback inside the LNLM fitting loop. Check the actual count before
# treating the fix as necessary -- if 0, the per-window fill computed
# below is a no-op in practice, but is still implemented correctly
# (not skipped) since a future dataset could have nonzero counts.

print("\n" + "=" * 70)
print("DIAGNOSTIC: NaN entries in RMSE over surviving-eligible factors")
print("=" * 70)

# combined_feature_idx (all features with valid LNLM params) is used
# here rather than combined_surviving (not computed until Part C) --
# close enough for this diagnostic's purpose, and avoids a forward
# dependency on Part C.
for w in EVAL_WINDOWS:
    n_nan = np.isnan(rmse[w][:, combined_feature_idx]).sum()
    pct = n_nan / (n_months * len(combined_feature_idx)) * 100
    print(f"  rmse[{w}] ({WINDOW_NAMES[w]}): {n_nan} NaN entries "
          f"({pct:.4f}% of {n_months} months x {len(combined_feature_idx)} factors)")

print("\n  If all three counts are 0 (or negligibly small), the per-window")
print("  fill computed in Part E is a no-op in practice for this dataset --")
print("  implemented anyway for correctness, not skipped.")


# ==========================================================================
# %% [markdown]
# ## Part C: Apply the 80% Noise Filter

# %%
avg_rmse_full = np.nanmean(rmse[5], axis=0)

subset_rmses = avg_rmse_full[combined_feature_idx]
valid_mask = ~np.isnan(subset_rmses)
valid_idx = combined_feature_idx[valid_mask]
valid_rmses = subset_rmses[valid_mask]

threshold_80 = np.percentile(valid_rmses, 80)
combined_surviving = valid_idx[valid_rmses <= threshold_80]

print(f"80% Noise Filter (based on 1-week RMSE):")
print(f"  Features with valid params:  {len(combined_feature_idx)}")
print(f"  Features with valid RMSE:    {len(valid_idx)}")
print(f"  After 80% filter:            {len(combined_surviving)}")
print(f"  RMSE threshold:              {threshold_80:.6f}")


# ==========================================================================
# %% [markdown]
# ## Part D: Compute Monthly Forward Returns

# %%
log_ret = np.log1p(daily_returns)
cum_log = np.cumsum(log_ret)

monthly_fwd_63d = np.full(n_months, np.nan)
for m in range(n_months):
    idx = month_rows[m]
    if idx + 63 < len(cum_log):
        monthly_fwd_63d[m] = np.expm1(cum_log[idx + 63] - cum_log[idx])

n_classifiable = np.sum(~np.isnan(monthly_fwd_63d))
n_precrisis = np.sum(monthly_fwd_63d < 0)

print(f"Monthly forward returns:")
print(f"  Classifiable: {n_classifiable}/{n_months}")
print(f"  Pre-crisis:   {n_precrisis} ({n_precrisis/n_classifiable:.1%})")


# ==========================================================================
# %% [markdown]
# ## Part E: Compute Hellinger Distance (Aggregate) + Shared Per-Window Imputation

# %%
ALPHA_MONTHLY = np.exp(-np.log(2) / 120)
FWD_LAG = 3
alpha_lag = ALPHA_MONTHLY ** FWD_LAG


def compute_global_fill(rmse_matrix, surviving_idx):
    """
    ONE fill value per (month, surviving factor), from the FULL surviving
    set, for THIS rmse_matrix (i.e. this specific evaluation window).
    Must be computed separately per window -- see header note on why a
    single 5-day-derived fill is not valid for the 1-day/10-day windows.
    """
    rmse_sub_all = rmse_matrix[:, surviving_idx]
    fill = np.full((rmse_sub_all.shape[0], rmse_sub_all.shape[1]), 0.0)
    for m in range(rmse_sub_all.shape[0]):
        row = rmse_sub_all[m]
        med = np.nanmedian(row)
        fill[m, :] = med if not np.isnan(med) else 0.0
    return fill


def compute_hellinger(rmse_matrix, surviving_idx, fwd_returns, fill_matrix,
                      subset_positions=None):
    """
    fill_matrix must be the fill array computed FOR THIS SAME rmse_matrix
    (i.e. this same evaluation window) via compute_global_fill -- not a
    fill array computed from a different window's RMSE.
    """
    n_surv = len(surviving_idx)
    rmse_sub = rmse_matrix[:, surviving_idx]

    if subset_positions is None:
        fill_sub = fill_matrix
    else:
        fill_sub = fill_matrix[:, subset_positions]

    pc_num = np.zeros(n_surv); pc_den = 0.0
    nt_num = np.zeros(n_surv); nt_den = 0.0

    H = np.full(n_months, np.nan)
    SRI = np.full(n_months, np.nan)

    for m in range(n_months):
        pc_num *= ALPHA_MONTHLY; pc_den *= ALPHA_MONTHLY
        nt_num *= ALPHA_MONTHLY; nt_den *= ALPHA_MONTHLY

        row = rmse_sub[m].copy()
        nan_mask = np.isnan(row)
        if nan_mask.any():
            row[nan_mask] = fill_sub[m][nan_mask]

        nt_num += row; nt_den += 1.0

        m_new = m - FWD_LAG
        if m_new >= 0 and not np.isnan(fwd_returns[m_new]):
            if fwd_returns[m_new] < 0:
                severity = fwd_returns[m_new] ** 2
                weight = severity * alpha_lag
                pc_row = rmse_sub[m_new].copy()
                pc_nan = np.isnan(pc_row)
                if pc_nan.any():
                    pc_row[pc_nan] = fill_sub[m_new][pc_nan]
                pc_num += weight * pc_row; pc_den += weight

        if pc_den > 1e-20:
            R_pre = pc_num / pc_den
            diff = np.sqrt(row) - np.sqrt(R_pre)
            H[m] = (1.0 / np.sqrt(2)) * np.sqrt(np.sum(diff ** 2))
            if m > 0 and not np.isnan(H[m - 1]):
                SRI[m] = 1.0 if H[m] < H[m - 1] else 0.0

    pc_ref = pc_num / pc_den if pc_den > 1e-20 else None
    nt_ref = nt_num / nt_den if nt_den > 1e-20 else None

    return {"H": H, "SRI": SRI, "pc_ref": pc_ref, "nt_ref": nt_ref}


# ── Compute the shared fill array ONCE PER WINDOW, over the full
# surviving set -- fixes the bug where a single 5-day fill was reused
# for the 1-day and 10-day computations. ──
GLOBAL_FILL = {w: compute_global_fill(rmse[w], combined_surviving) for w in EVAL_WINDOWS}
print("Global fill arrays computed, one per evaluation window:")
for w in EVAL_WINDOWS:
    print(f"  window={w} ({WINDOW_NAMES[w]}): shape {GLOBAL_FILL[w].shape}")
print(f"  These SAME per-window arrays are saved for Notebook 3's "
      f"per-theme computations.")

# %%
print("\nComputing Hellinger distance for all evaluation windows (aggregate)...")
results = {}
for w in EVAL_WINDOWS:
    results[w] = compute_hellinger(rmse[w], combined_surviving, monthly_fwd_63d,
                                    GLOBAL_FILL[w], subset_positions=None)

print("  Done.")


# ==========================================================================
# %% [markdown]
# ## Part F: Significance Tests

# %%
# ── F.1: Evaluation window sensitivity ──

print("=" * 95)
print("TABLE 1: EVALUATION WINDOW SENSITIVITY (Combined Dataset)")
print("  Monthly: n ~ 134 (some overlap in 63-day forward returns)")
print("  Quarterly: n ~ 45 (zero overlap, average of 3 offsets)")
print("  Stars: *** p<0.01  ** p<0.05  * p<0.10")
print("=" * 95)

print(f"\n  {'Window':12s} {'Monthly t':>10s} {'Monthly p':>10s} "
      f"{'Quarterly t':>12s} {'Quarterly p':>12s} {'H range':>20s}")
print(f"  {'-' * 80}")

for w in EVAL_WINDOWS:
    H = results[w]["H"]

    lg_m = logistic_t(H, monthly_fwd_63d)

    q_ts = []; q_ps = []
    for offset in range(3):
        q_idx = np.arange(offset, n_months, 3)
        lg_q = logistic_t(H[q_idx], monthly_fwd_63d[q_idx])
        if lg_q:
            q_ts.append(lg_q["t"]); q_ps.append(lg_q["p"])

    m_t = f"{lg_m['t']:+10.3f}" if lg_m else "       N/A"
    m_p = f"{lg_m['p']:10.4f}" if lg_m else "       N/A"

    if len(q_ts) == 3:
        avg_qt = np.mean(q_ts); avg_qp = np.mean(q_ps)
        qt_str = f"{avg_qt:+12.3f}"; qp_str = f"{avg_qp:12.4f}"
        star = ("***" if avg_qp < 0.01 else "**" if avg_qp < 0.05
                else "*" if avg_qp < 0.10 else "")
    else:
        qt_str = "         N/A"; qp_str = "         N/A"; star = ""

    h_min = np.nanmin(H); h_max = np.nanmax(H)
    h_range = f"[{h_min:.3f}, {h_max:.3f}]"

    print(f"  {WINDOW_NAMES[w]:12s} {m_t} {m_p} {qt_str} {qp_str} "
          f"{h_range:>20s} {star}")

# %%
# ── F.2: Noise filter sensitivity ──

print(f"\n{'=' * 80}")
print("TABLE 2: NOISE FILTER SENSITIVITY (5-day window, combined)")
print("  Varies the percentage of lowest-RMSE factors retained.")
print("=" * 80)

print(f"\n  {'Filter':>7s} {'n_fac':>6s} {'Monthly t':>10s} {'Monthly p':>10s} "
      f"{'Quarterly t':>12s} {'Quarterly p':>12s}")
print(f"  {'-' * 62}")

for pct in [1.00, 0.90, 0.80, 0.70, 0.60, 0.50, 0.40, 0.30, 0.20, 0.10, 0.05, 0.01]:
    threshold = np.percentile(valid_rmses, pct * 100)
    surviving = valid_idx[valid_rmses <= threshold]

    if len(surviving) < 10:
        continue

    # Fresh per-window fill for this varying surviving set (5-day window
    # only, since that's all this table tests) -- correct since the
    # surviving SET changes at each threshold.
    fill_this = compute_global_fill(rmse[5], surviving)
    r = compute_hellinger(rmse[5], surviving, monthly_fwd_63d,
                          fill_this, subset_positions=None)

    lg_m = logistic_t(r["H"], monthly_fwd_63d)

    q_ts = []; q_ps = []
    for offset in range(3):
        q_idx = np.arange(offset, n_months, 3)
        lg_q = logistic_t(r["H"][q_idx], monthly_fwd_63d[q_idx])
        if lg_q:
            q_ts.append(lg_q["t"]); q_ps.append(lg_q["p"])

    m_t = f"{lg_m['t']:+10.3f}" if lg_m else "       N/A"
    m_p = f"{lg_m['p']:10.4f}" if lg_m else "       N/A"

    if len(q_ts) == 3:
        avg_qt = np.mean(q_ts); avg_qp = np.mean(q_ps)
        qt_str = f"{avg_qt:+12.3f}"; qp_str = f"{avg_qp:12.4f}"
        star = ("***" if avg_qp < 0.01 else "**" if avg_qp < 0.05
                else "*" if avg_qp < 0.10 else "")
    else:
        qt_str = "         N/A"; qp_str = "         N/A"; star = ""

    print(f"  {pct:7.0%} {len(surviving):6d} {m_t} {m_p} {qt_str} {qp_str} {star}")

# %%
# ── F.3: Autocorrelation diagnostic ──

print(f"\n{'=' * 60}")
print("AUTOCORRELATION OF 3-MONTH FORWARD RETURNS (monthly sampling)")
print("=" * 60)

fwd_valid = monthly_fwd_63d[~np.isnan(monthly_fwd_63d)]

print(f"\n  {'Lag':>12s} {'rho':>8s} {'Overlap':>25s}")
print(f"  {'-' * 48}")
for lag, overlap in [(1, "~42/63 days shared"),
                      (2, "~21/63 days shared"),
                      (3, "~0/63 days shared")]:
    if len(fwd_valid) > lag:
        rho = np.corrcoef(fwd_valid[:-lag], fwd_valid[lag:])[0, 1]
        print(f"  {lag:9d} mo {rho:+8.4f} {overlap:>25s}")

autocorr_sum = 0
for lag in range(1, min(20, len(fwd_valid))):
    rho = np.corrcoef(fwd_valid[:-lag], fwd_valid[lag:])[0, 1]
    if abs(rho) < 0.05:
        break
    autocorr_sum += rho

n_eff = len(fwd_valid) / (1 + 2 * autocorr_sum)
print(f"\n  Nominal n (monthly):    {len(fwd_valid)}")
print(f"  Effective n (adjusted): {n_eff:.0f}")
print(f"  Inflation ratio:        {len(fwd_valid)/n_eff:.1f}x")
print(f"\n  The quarterly test (n ~ 45, zero overlap) avoids this issue entirely.")


# ==========================================================================
# %% [markdown]
# ## Part G: Save Everything

# %%
# ── G.1: Save RMSE matrices ──
rmse_save = {"dates": param_dates, "feature_names": np.array(feature_names)}
for w in EVAL_WINDOWS:
    rmse_save[f"rmse_{w}d"] = rmse[w]
np.savez(DATA_PM / f"rmse_matrices_{DATASET_NAME}.npz", **rmse_save)
print(f"RMSE matrices saved ({len(EVAL_WINDOWS)} windows)")

# %%
# ── G.2: Save Hellinger distances, SRI, reference vectors, AND one
# per-window shared imputation array. Notebook 3 MUST use
# global_fill_{w}d when computing a per-theme H at window w -- reusing
# a different window's fill array would reintroduce the bug fixed here. ──

hellinger_save = {
    "dates": param_dates,
    "monthly_fwd_63d": monthly_fwd_63d,
    "combined_surviving_idx": combined_surviving,
}

for w in EVAL_WINDOWS:
    hellinger_save[f"global_fill_{w}d"] = GLOBAL_FILL[w]

    r = results[w]
    hellinger_save[f"H_{w}d"] = r["H"]
    hellinger_save[f"SRI_{w}d"] = r["SRI"]
    if r["pc_ref"] is not None:
        hellinger_save[f"pcref_{w}d"] = r["pc_ref"]
    if r["nt_ref"] is not None:
        hellinger_save[f"ntref_{w}d"] = r["nt_ref"]

np.savez(DATA_PM / f"hellinger_monthly_{DATASET_NAME}.npz", **hellinger_save)
print(f"Hellinger series saved ({len(EVAL_WINDOWS)} windows), "
      f"including one shared imputation array PER WINDOW for Notebook 3")

# %%
# ── G.3: Re-save target ──

target_save = pd.DataFrame({
    "date": pd.DatetimeIndex(all_dates),
    "daily_return": daily_returns,
    "monthly_return": monthly_target,
})
target_save.to_parquet(DATA_PM / f"polymodel_target_{DATASET_NAME}.parquet", index=False)
print(f"Target re-saved: {target_save.shape}")

print(f"\n{'=' * 60}")
print("COMPLETE")
print(f"  RMSE:      {DATA_PM / f'rmse_matrices_{DATASET_NAME}.npz'}")
print(f"  Hellinger: {DATA_PM / f'hellinger_monthly_{DATASET_NAME}.npz'}")
print(f"  Target:    {DATA_PM / f'polymodel_target_{DATASET_NAME}.parquet'}")
print(f"{'=' * 60}")

DATASET_NAME: agg_full_moments
Reading/writing from: ..\..\Data\Results\Reproducing_Hellinger_Polymodel\intermediate
Parameters loaded: 149 months x 1699 features
  ✓ Feature column alignment confirmed (1699 features)
Features: (4384, 1699)
Target:   4384 rows
Features lagged by 21 days. First valid row: 21

Combined dataset: 1691 features with valid parameters
  (out of 1699 total, 8 cleaned out)

Evaluation windows: ['2 weeks', '1 week', '1 day']
Estimation dates: 149 months (2012-08 to 2024-12)
Computing RMSE: 1699 features x 149 months x 3 windows...


C:\Users\Henry\AppData\Local\Temp\ipykernel_28320\2464390916.py:220: RuntimeWarning: Mean of empty slice
  rmse[w][m] = np.sqrt(np.nanmean(sq_resid[-w:], axis=0))


  Month 20/149  [6s elapsed, ~40s remaining]
  Month 40/149  [11s elapsed, ~30s remaining]
  Month 60/149  [15s elapsed, ~23s remaining]
  Month 80/149  [20s elapsed, ~18s remaining]
  Month 100/149  [26s elapsed, ~13s remaining]
  Month 120/149  [30s elapsed, ~7s remaining]
  Month 140/149  [35s elapsed, ~2s remaining]
  Done in 37.2s

  Mean RMSE by evaluation window:
    2 weeks     : 0.033343
    1 week      : 0.032906
    1 day       : 0.032270

DIAGNOSTIC: NaN entries in RMSE over surviving-eligible factors
  rmse[10] (2 weeks): 0 NaN entries (0.0000% of 149 months x 1691 factors)
  rmse[5] (1 week): 0 NaN entries (0.0000% of 149 months x 1691 factors)
  rmse[1] (1 day): 0 NaN entries (0.0000% of 149 months x 1691 factors)

  If all three counts are 0 (or negligibly small), the per-window
  fill computed in Part E is a no-op in practice for this dataset --
  implemented anyway for correctness, not skipped.
80% Noise Filter (based on 1-week RMSE):
  Features with valid params:  16

C:\Users\Henry\AppData\Local\Temp\ipykernel_28320\2464390916.py:269: RuntimeWarning: Mean of empty slice
  avg_rmse_full = np.nanmean(rmse[5], axis=0)


     100%   1691     +0.699     0.4844       +0.461       0.4619 
      90%   1522     +0.694     0.4875       +0.459       0.4587 
      80%   1353     +0.691     0.4897       +0.457       0.4547 
      70%   1184     +0.689     0.4911       +0.457       0.4511 
      60%   1015     +0.693     0.4884       +0.461       0.4450 
      50%    846     +0.700     0.4837       +0.466       0.4382 
      40%    677     +0.698     0.4851       +0.466       0.4350 
      30%    508     +0.676     0.4992       +0.455       0.4269 
      20%    339     +0.670     0.5032       +0.454       0.4257 
      10%    170     +0.653     0.5139       +0.449       0.4276 
       5%     85     +0.655     0.5122       +0.459       0.4165 
       1%     17     +0.676     0.4990       +0.482       0.4380 

AUTOCORRELATION OF 3-MONTH FORWARD RETURNS (monthly sampling)

           Lag      rho                   Overlap
  ------------------------------------------------
          1 mo  +0.4852        ~42/63 days 

In [1]:
# %% [markdown]
# # RMSE and Hellinger Distance Computation (Stage 5 Pipeline)
#
# Takes the LNLM parameters fitted in Notebook 1 (Stage 5 data source)
# and:
#
# 1. Evaluates each factor's model on the estimation window
# 2. Computes the RMSE over multiple evaluation windows
# 3. Applies the 80% noise filter
# 4. Computes the monthly Hellinger distance and SRI (aggregate/combined)
# 5. Tests significance with both monthly and quarterly sampling
# 6. Checks robustness to filter threshold
# 7. Saves everything Notebook 3 needs -- including RMSE matrices AND a
#    shared per-month, PER-WINDOW imputation array, so the theme-level
#    H^2 identity in Notebook 3 holds EXACTLY, not just approximately.
#
# ═══════════════════════════════════════════════════════════════════════
# WHY THE IMPUTATION FIX MATTERS (and why it's now per-window)
# ═══════════════════════════════════════════════════════════════════════
#
# The old notebook's compute_hellinger() imputed a missing factor's RMSE
# for a given month using np.nanmedian() computed over WHATEVER SUBSET
# of factors was passed in -- the full surviving set for the aggregate
# H, but a single theme's factors for a per-theme H. A factor with a
# missing month got a DIFFERENT imputed value depending on which grouping
# it was summed within, which breaks
#     H_aggregate^2 = sum_over_themes( H_theme^2 )
#
# FIX (v1, corrected here to v2): compute ONE fill value per (month,
# surviving factor), from the FULL surviving set, ONCE -- but do this
# SEPARATELY for each evaluation window (1-day, 5-day, 10-day), not just
# once from the 5-day RMSE and reused everywhere. A NaN in rmse[1] (the
# 1-day window) represents a missing 1-day RMSE; filling it with a value
# drawn from the 5-day RMSE distribution for that month is not just
# "a different grouping's median" (which the v1 fix already handled) --
# it is a different, unrelated quantity with a different scale. The H^2
# identity would still technically hold under the v1 (single-window)
# fix, since the SAME wrong value would be reused consistently across
# any aggregate-vs-theme comparison at a given window -- but the
# substituted number itself would misrepresent what "typical RMSE at
# this window, this month" means for the 1-day and 10-day windows
# specifically.
#
# A diagnostic below (Part B.1) checks whether this actually matters in
# practice for this dataset: given the Notebook 1 pre-flight already
# confirmed zero NaNs in the raw feature panel and target, a NaN can only
# appear in rmse[w][m,f] for a SURVIVING factor via an occasional
# single-month std<1e-10 or valid.sum()<100 fallback inside the fitting
# loop -- plausibly rare or absent. If the diagnostic count is 0, the
# per-window fix is a no-op in practice; it is still implemented properly
# below (not skipped) so this notebook is correct regardless of what any
# future dataset's diagnostic shows.
#
# ═══════════════════════════════════════════════════════════════════════
DATASET_NAME = "agg_means"   # or "agg_means"
# ═══════════════════════════════════════════════════════════════════════

# ==========================================================================
# %% [markdown]
# ## Part A: Setup and Data Loading

# %%
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import norm as normal_dist
from pathlib import Path
import time

ROOT = Path("../..")
RESULTS_ROOT = ROOT / "Data" / "Results" / "Reproducing_Hellinger_Polymodel"
DATA_PM = RESULTS_ROOT / "intermediate"

print(f"DATASET_NAME: {DATASET_NAME}")
print(f"Reading/writing from: {DATA_PM}")

# %%
# ── A.1: Load LNLM parameters from Notebook 1 ──

params_path = DATA_PM / f"lnlm_params_{DATASET_NAME}.npz"
assert params_path.exists(), (
    f"Not found: {params_path} -- run Notebook 1 with DATASET_NAME="
    f"'{DATASET_NAME}' first."
)
params = np.load(params_path, allow_pickle=True)

beta_lin    = params["beta_lin"]
beta_nonlin = params["beta_nonlin"]
mu_star     = params["mu_star"]
y_mean      = params["y_mean"]
param_dates = params["dates"]
feature_names = list(params["feature_names"])

n_months = len(param_dates)
n_features = len(feature_names)

print(f"Parameters loaded: {n_months} months x {n_features} features")

# %%
# ── A.2: Load features (unlagged) and target ──

features_path = DATA_PM / f"polymodel_features_unlagged_{DATASET_NAME}.parquet"
target_path   = DATA_PM / f"polymodel_target_{DATASET_NAME}.parquet"

features_df = pd.read_parquet(features_path)
target_df   = pd.read_parquet(target_path)

all_dates      = pd.to_datetime(features_df["date"]).values
feature_cols   = [c for c in features_df.columns if c != "date"]
features_raw   = features_df[feature_cols].values.astype(np.float64)
monthly_target = target_df["monthly_return"].values.astype(np.float64)
daily_returns  = target_df["daily_return"].values.astype(np.float64)

assert feature_cols == feature_names, (
    "Feature columns in the unlagged parquet do not match feature_names "
    "in the parameter file -- these must come from the SAME Notebook 1 "
    "run for the same DATASET_NAME. STOP."
)
print(f"  ✓ Feature column alignment confirmed ({len(feature_cols)} features)")

print(f"Features: {features_raw.shape}")
print(f"Target:   {len(monthly_target)} rows")

# %%
# ── A.3: Apply the 21-day lag ──

LAG = 21
feature_values = np.full_like(features_raw, np.nan)
feature_values[LAG:] = features_raw[:-LAG]

print(f"Features lagged by {LAG} days. First valid row: {LAG}")

# %%
# ── A.4: Load metadata ──

meta_path = DATA_PM / f"polymodel_feature_meta_{DATASET_NAME}.csv"
meta = pd.read_csv(meta_path, dtype={"subtheme_id": str, "theme_id": str})

has_valid_params = np.any(~np.isnan(mu_star), axis=0)
combined_feature_idx = np.where(has_valid_params)[0]

print(f"\nCombined dataset: {len(combined_feature_idx)} features with valid parameters")
print(f"  (out of {n_features} total, {n_features - len(combined_feature_idx)} cleaned out)")

# %%
# ── A.5: Define evaluation windows and month-end row indices ──

WINDOW = 1260

EVAL_WINDOWS = [10, 5, 1]
WINDOW_NAMES = {
    10:   "2 weeks",
    5:    "1 week",
    1:    "1 day",
}

month_rows = np.array([
    int(np.searchsorted(all_dates, d)) for d in param_dates
])

print(f"\nEvaluation windows: {[WINDOW_NAMES[w] for w in EVAL_WINDOWS]}")
print(f"Estimation dates: {n_months} months "
      f"({pd.Timestamp(param_dates[0]).strftime('%Y-%m')} to "
      f"{pd.Timestamp(param_dates[-1]).strftime('%Y-%m')})")

# %%
# ── A.6: Logistic regression helper ──

def logistic_t(signal, fwd_return, label=""):
    valid = ~np.isnan(signal) & ~np.isnan(fwd_return)
    if valid.sum() < 30:
        return None
    y = (fwd_return[valid] > 0).astype(float)
    X = sm.add_constant(signal[valid])
    try:
        fit = sm.Logit(y, X).fit(disp=0)
        return {"label": label, "n": int(valid.sum()),
                "t": fit.tvalues[1], "p": fit.pvalues[1]}
    except Exception:
        return None


# ==========================================================================
# %% [markdown]
# ## Part B: Compute RMSE Across All Evaluation Windows

# %%
rmse = {}
for w in EVAL_WINDOWS:
    rmse[w] = np.full((n_months, n_features), np.nan)

print(f"Computing RMSE: {n_features} features x {n_months} months "
      f"x {len(EVAL_WINDOWS)} windows...")
t0 = time.perf_counter()

for m in range(n_months):
    idx = month_rows[m]
    ws = idx - WINDOW + 1
    we = idx + 1

    y = monthly_target[ws:we]
    X = feature_values[ws:we]
    ym = y_mean[m]

    bl = beta_lin[m]
    bn = beta_nonlin[m]
    ms = mu_star[m]

    X2 = X * X
    nonlin_pred = (X * bn[:, 0] +
                   (X2 - 1) * bn[:, 1] +
                   (X2 * X - 3 * X) * bn[:, 2] +
                   (X2 * X2 - 6 * X2 + 3) * bn[:, 3])
    lin_pred = X * bl
    y_hat = ym + ms * nonlin_pred + (1 - ms) * lin_pred

    sq_resid = (y[:, np.newaxis] - y_hat) ** 2

    for w in EVAL_WINDOWS:
        rmse[w][m] = np.sqrt(np.nanmean(sq_resid[-w:], axis=0))

    if (m + 1) % 20 == 0:
        elapsed = time.perf_counter() - t0
        eta = elapsed / (m + 1) * (n_months - m - 1)
        print(f"  Month {m+1}/{n_months}  [{elapsed:.0f}s elapsed, ~{eta:.0f}s remaining]")

print(f"  Done in {time.perf_counter() - t0:.1f}s")

print(f"\n  Mean RMSE by evaluation window:")
for w in EVAL_WINDOWS:
    print(f"    {WINDOW_NAMES[w]:12s}: {np.nanmean(rmse[w]):.6f}")

# %%
# ── B.1: DIAGNOSTIC -- does the imputation mismatch actually matter here?
#
# Given Notebook 1's pre-flight already confirmed zero NaNs in the raw
# feature panel and target, a NaN in rmse[w][m,f] for a SURVIVING factor
# (one that already passed the >=50%-valid-months cleanup criterion) can
# only arise from an occasional single-month std<1e-10 or valid.sum()<100
# fallback inside the LNLM fitting loop. Check the actual count before
# treating the fix as necessary -- if 0, the per-window fill computed
# below is a no-op in practice, but is still implemented correctly
# (not skipped) since a future dataset could have nonzero counts.

print("\n" + "=" * 70)
print("DIAGNOSTIC: NaN entries in RMSE over surviving-eligible factors")
print("=" * 70)

# combined_feature_idx (all features with valid LNLM params) is used
# here rather than combined_surviving (not computed until Part C) --
# close enough for this diagnostic's purpose, and avoids a forward
# dependency on Part C.
for w in EVAL_WINDOWS:
    n_nan = np.isnan(rmse[w][:, combined_feature_idx]).sum()
    pct = n_nan / (n_months * len(combined_feature_idx)) * 100
    print(f"  rmse[{w}] ({WINDOW_NAMES[w]}): {n_nan} NaN entries "
          f"({pct:.4f}% of {n_months} months x {len(combined_feature_idx)} factors)")

print("\n  If all three counts are 0 (or negligibly small), the per-window")
print("  fill computed in Part E is a no-op in practice for this dataset --")
print("  implemented anyway for correctness, not skipped.")


# ==========================================================================
# %% [markdown]
# ## Part C: Apply the 80% Noise Filter

# %%
avg_rmse_full = np.nanmean(rmse[5], axis=0)

subset_rmses = avg_rmse_full[combined_feature_idx]
valid_mask = ~np.isnan(subset_rmses)
valid_idx = combined_feature_idx[valid_mask]
valid_rmses = subset_rmses[valid_mask]

threshold_80 = np.percentile(valid_rmses, 80)
combined_surviving = valid_idx[valid_rmses <= threshold_80]

print(f"80% Noise Filter (based on 1-week RMSE):")
print(f"  Features with valid params:  {len(combined_feature_idx)}")
print(f"  Features with valid RMSE:    {len(valid_idx)}")
print(f"  After 80% filter:            {len(combined_surviving)}")
print(f"  RMSE threshold:              {threshold_80:.6f}")


# ==========================================================================
# %% [markdown]
# ## Part D: Compute Monthly Forward Returns

# %%
log_ret = np.log1p(daily_returns)
cum_log = np.cumsum(log_ret)

monthly_fwd_63d = np.full(n_months, np.nan)
for m in range(n_months):
    idx = month_rows[m]
    if idx + 63 < len(cum_log):
        monthly_fwd_63d[m] = np.expm1(cum_log[idx + 63] - cum_log[idx])

n_classifiable = np.sum(~np.isnan(monthly_fwd_63d))
n_precrisis = np.sum(monthly_fwd_63d < 0)

print(f"Monthly forward returns:")
print(f"  Classifiable: {n_classifiable}/{n_months}")
print(f"  Pre-crisis:   {n_precrisis} ({n_precrisis/n_classifiable:.1%})")


# ==========================================================================
# %% [markdown]
# ## Part E: Compute Hellinger Distance (Aggregate) + Shared Per-Window Imputation

# %%
ALPHA_MONTHLY = np.exp(-np.log(2) / 120)
FWD_LAG = 3
alpha_lag = ALPHA_MONTHLY ** FWD_LAG


def compute_global_fill(rmse_matrix, surviving_idx):
    """
    ONE fill value per (month, surviving factor), from the FULL surviving
    set, for THIS rmse_matrix (i.e. this specific evaluation window).
    Must be computed separately per window -- see header note on why a
    single 5-day-derived fill is not valid for the 1-day/10-day windows.
    """
    rmse_sub_all = rmse_matrix[:, surviving_idx]
    fill = np.full((rmse_sub_all.shape[0], rmse_sub_all.shape[1]), 0.0)
    for m in range(rmse_sub_all.shape[0]):
        row = rmse_sub_all[m]
        med = np.nanmedian(row)
        fill[m, :] = med if not np.isnan(med) else 0.0
    return fill


def compute_hellinger(rmse_matrix, surviving_idx, fwd_returns, fill_matrix,
                      subset_positions=None):
    """
    fill_matrix must be the fill array computed FOR THIS SAME rmse_matrix
    (i.e. this same evaluation window) via compute_global_fill -- not a
    fill array computed from a different window's RMSE.
    """
    n_surv = len(surviving_idx)
    rmse_sub = rmse_matrix[:, surviving_idx]

    if subset_positions is None:
        fill_sub = fill_matrix
    else:
        fill_sub = fill_matrix[:, subset_positions]

    pc_num = np.zeros(n_surv); pc_den = 0.0
    nt_num = np.zeros(n_surv); nt_den = 0.0

    H = np.full(n_months, np.nan)
    SRI = np.full(n_months, np.nan)

    for m in range(n_months):
        pc_num *= ALPHA_MONTHLY; pc_den *= ALPHA_MONTHLY
        nt_num *= ALPHA_MONTHLY; nt_den *= ALPHA_MONTHLY

        row = rmse_sub[m].copy()
        nan_mask = np.isnan(row)
        if nan_mask.any():
            row[nan_mask] = fill_sub[m][nan_mask]

        nt_num += row; nt_den += 1.0

        m_new = m - FWD_LAG
        if m_new >= 0 and not np.isnan(fwd_returns[m_new]):
            if fwd_returns[m_new] < 0:
                severity = fwd_returns[m_new] ** 2
                weight = severity * alpha_lag
                pc_row = rmse_sub[m_new].copy()
                pc_nan = np.isnan(pc_row)
                if pc_nan.any():
                    pc_row[pc_nan] = fill_sub[m_new][pc_nan]
                pc_num += weight * pc_row; pc_den += weight

        if pc_den > 1e-20:
            R_pre = pc_num / pc_den
            diff = np.sqrt(row) - np.sqrt(R_pre)
            H[m] = (1.0 / np.sqrt(2)) * np.sqrt(np.sum(diff ** 2))
            if m > 0 and not np.isnan(H[m - 1]):
                SRI[m] = 1.0 if H[m] < H[m - 1] else 0.0

    pc_ref = pc_num / pc_den if pc_den > 1e-20 else None
    nt_ref = nt_num / nt_den if nt_den > 1e-20 else None

    return {"H": H, "SRI": SRI, "pc_ref": pc_ref, "nt_ref": nt_ref}


# ── Compute the shared fill array ONCE PER WINDOW, over the full
# surviving set -- fixes the bug where a single 5-day fill was reused
# for the 1-day and 10-day computations. ──
GLOBAL_FILL = {w: compute_global_fill(rmse[w], combined_surviving) for w in EVAL_WINDOWS}
print("Global fill arrays computed, one per evaluation window:")
for w in EVAL_WINDOWS:
    print(f"  window={w} ({WINDOW_NAMES[w]}): shape {GLOBAL_FILL[w].shape}")
print(f"  These SAME per-window arrays are saved for Notebook 3's "
      f"per-theme computations.")

# %%
print("\nComputing Hellinger distance for all evaluation windows (aggregate)...")
results = {}
for w in EVAL_WINDOWS:
    results[w] = compute_hellinger(rmse[w], combined_surviving, monthly_fwd_63d,
                                    GLOBAL_FILL[w], subset_positions=None)

print("  Done.")


# ==========================================================================
# %% [markdown]
# ## Part F: Significance Tests

# %%
# ── F.1: Evaluation window sensitivity ──

print("=" * 95)
print("TABLE 1: EVALUATION WINDOW SENSITIVITY (Combined Dataset)")
print("  Monthly: n ~ 134 (some overlap in 63-day forward returns)")
print("  Quarterly: n ~ 45 (zero overlap, average of 3 offsets)")
print("  Stars: *** p<0.01  ** p<0.05  * p<0.10")
print("=" * 95)

print(f"\n  {'Window':12s} {'Monthly t':>10s} {'Monthly p':>10s} "
      f"{'Quarterly t':>12s} {'Quarterly p':>12s} {'H range':>20s}")
print(f"  {'-' * 80}")

for w in EVAL_WINDOWS:
    H = results[w]["H"]

    lg_m = logistic_t(H, monthly_fwd_63d)

    q_ts = []; q_ps = []
    for offset in range(3):
        q_idx = np.arange(offset, n_months, 3)
        lg_q = logistic_t(H[q_idx], monthly_fwd_63d[q_idx])
        if lg_q:
            q_ts.append(lg_q["t"]); q_ps.append(lg_q["p"])

    m_t = f"{lg_m['t']:+10.3f}" if lg_m else "       N/A"
    m_p = f"{lg_m['p']:10.4f}" if lg_m else "       N/A"

    if len(q_ts) == 3:
        avg_qt = np.mean(q_ts); avg_qp = np.mean(q_ps)
        qt_str = f"{avg_qt:+12.3f}"; qp_str = f"{avg_qp:12.4f}"
        star = ("***" if avg_qp < 0.01 else "**" if avg_qp < 0.05
                else "*" if avg_qp < 0.10 else "")
    else:
        qt_str = "         N/A"; qp_str = "         N/A"; star = ""

    h_min = np.nanmin(H); h_max = np.nanmax(H)
    h_range = f"[{h_min:.3f}, {h_max:.3f}]"

    print(f"  {WINDOW_NAMES[w]:12s} {m_t} {m_p} {qt_str} {qp_str} "
          f"{h_range:>20s} {star}")

# %%
# ── F.2: Noise filter sensitivity ──

print(f"\n{'=' * 80}")
print("TABLE 2: NOISE FILTER SENSITIVITY (5-day window, combined)")
print("  Varies the percentage of lowest-RMSE factors retained.")
print("=" * 80)

print(f"\n  {'Filter':>7s} {'n_fac':>6s} {'Monthly t':>10s} {'Monthly p':>10s} "
      f"{'Quarterly t':>12s} {'Quarterly p':>12s}")
print(f"  {'-' * 62}")

for pct in [1.00, 0.90, 0.80, 0.70, 0.60, 0.50, 0.40, 0.30, 0.20, 0.10, 0.05, 0.01]:
    threshold = np.percentile(valid_rmses, pct * 100)
    surviving = valid_idx[valid_rmses <= threshold]

    if len(surviving) < 10:
        continue

    # Fresh per-window fill for this varying surviving set (5-day window
    # only, since that's all this table tests) -- correct since the
    # surviving SET changes at each threshold.
    fill_this = compute_global_fill(rmse[5], surviving)
    r = compute_hellinger(rmse[5], surviving, monthly_fwd_63d,
                          fill_this, subset_positions=None)

    lg_m = logistic_t(r["H"], monthly_fwd_63d)

    q_ts = []; q_ps = []
    for offset in range(3):
        q_idx = np.arange(offset, n_months, 3)
        lg_q = logistic_t(r["H"][q_idx], monthly_fwd_63d[q_idx])
        if lg_q:
            q_ts.append(lg_q["t"]); q_ps.append(lg_q["p"])

    m_t = f"{lg_m['t']:+10.3f}" if lg_m else "       N/A"
    m_p = f"{lg_m['p']:10.4f}" if lg_m else "       N/A"

    if len(q_ts) == 3:
        avg_qt = np.mean(q_ts); avg_qp = np.mean(q_ps)
        qt_str = f"{avg_qt:+12.3f}"; qp_str = f"{avg_qp:12.4f}"
        star = ("***" if avg_qp < 0.01 else "**" if avg_qp < 0.05
                else "*" if avg_qp < 0.10 else "")
    else:
        qt_str = "         N/A"; qp_str = "         N/A"; star = ""

    print(f"  {pct:7.0%} {len(surviving):6d} {m_t} {m_p} {qt_str} {qp_str} {star}")

# %%
# ── F.3: Autocorrelation diagnostic ──

print(f"\n{'=' * 60}")
print("AUTOCORRELATION OF 3-MONTH FORWARD RETURNS (monthly sampling)")
print("=" * 60)

fwd_valid = monthly_fwd_63d[~np.isnan(monthly_fwd_63d)]

print(f"\n  {'Lag':>12s} {'rho':>8s} {'Overlap':>25s}")
print(f"  {'-' * 48}")
for lag, overlap in [(1, "~42/63 days shared"),
                      (2, "~21/63 days shared"),
                      (3, "~0/63 days shared")]:
    if len(fwd_valid) > lag:
        rho = np.corrcoef(fwd_valid[:-lag], fwd_valid[lag:])[0, 1]
        print(f"  {lag:9d} mo {rho:+8.4f} {overlap:>25s}")

autocorr_sum = 0
for lag in range(1, min(20, len(fwd_valid))):
    rho = np.corrcoef(fwd_valid[:-lag], fwd_valid[lag:])[0, 1]
    if abs(rho) < 0.05:
        break
    autocorr_sum += rho

n_eff = len(fwd_valid) / (1 + 2 * autocorr_sum)
print(f"\n  Nominal n (monthly):    {len(fwd_valid)}")
print(f"  Effective n (adjusted): {n_eff:.0f}")
print(f"  Inflation ratio:        {len(fwd_valid)/n_eff:.1f}x")
print(f"\n  The quarterly test (n ~ 45, zero overlap) avoids this issue entirely.")


# ==========================================================================
# %% [markdown]
# ## Part G: Save Everything

# %%
# ── G.1: Save RMSE matrices ──
rmse_save = {"dates": param_dates, "feature_names": np.array(feature_names)}
for w in EVAL_WINDOWS:
    rmse_save[f"rmse_{w}d"] = rmse[w]
np.savez(DATA_PM / f"rmse_matrices_{DATASET_NAME}.npz", **rmse_save)
print(f"RMSE matrices saved ({len(EVAL_WINDOWS)} windows)")

# %%
# ── G.2: Save Hellinger distances, SRI, reference vectors, AND one
# per-window shared imputation array. Notebook 3 MUST use
# global_fill_{w}d when computing a per-theme H at window w -- reusing
# a different window's fill array would reintroduce the bug fixed here. ──

hellinger_save = {
    "dates": param_dates,
    "monthly_fwd_63d": monthly_fwd_63d,
    "combined_surviving_idx": combined_surviving,
}

for w in EVAL_WINDOWS:
    hellinger_save[f"global_fill_{w}d"] = GLOBAL_FILL[w]

    r = results[w]
    hellinger_save[f"H_{w}d"] = r["H"]
    hellinger_save[f"SRI_{w}d"] = r["SRI"]
    if r["pc_ref"] is not None:
        hellinger_save[f"pcref_{w}d"] = r["pc_ref"]
    if r["nt_ref"] is not None:
        hellinger_save[f"ntref_{w}d"] = r["nt_ref"]

np.savez(DATA_PM / f"hellinger_monthly_{DATASET_NAME}.npz", **hellinger_save)
print(f"Hellinger series saved ({len(EVAL_WINDOWS)} windows), "
      f"including one shared imputation array PER WINDOW for Notebook 3")

# %%
# ── G.3: Re-save target ──

target_save = pd.DataFrame({
    "date": pd.DatetimeIndex(all_dates),
    "daily_return": daily_returns,
    "monthly_return": monthly_target,
})
target_save.to_parquet(DATA_PM / f"polymodel_target_{DATASET_NAME}.parquet", index=False)
print(f"Target re-saved: {target_save.shape}")

print(f"\n{'=' * 60}")
print("COMPLETE")
print(f"  RMSE:      {DATA_PM / f'rmse_matrices_{DATASET_NAME}.npz'}")
print(f"  Hellinger: {DATA_PM / f'hellinger_monthly_{DATASET_NAME}.npz'}")
print(f"  Target:    {DATA_PM / f'polymodel_target_{DATASET_NAME}.parquet'}")
print(f"{'=' * 60}")

DATASET_NAME: agg_means
Reading/writing from: ..\..\Data\Results\Reproducing_Hellinger_Polymodel\intermediate
Parameters loaded: 149 months x 574 features
  ✓ Feature column alignment confirmed (574 features)
Features: (4384, 574)
Target:   4384 rows
Features lagged by 21 days. First valid row: 21

Combined dataset: 573 features with valid parameters
  (out of 574 total, 1 cleaned out)

Evaluation windows: ['2 weeks', '1 week', '1 day']
Estimation dates: 149 months (2012-08 to 2024-12)
Computing RMSE: 574 features x 149 months x 3 windows...


C:\Users\Henry\AppData\Local\Temp\ipykernel_36112\1098613730.py:220: RuntimeWarning: Mean of empty slice
  rmse[w][m] = np.sqrt(np.nanmean(sq_resid[-w:], axis=0))


  Month 20/149  [1s elapsed, ~5s remaining]
  Month 40/149  [1s elapsed, ~4s remaining]
  Month 60/149  [2s elapsed, ~3s remaining]
  Month 80/149  [3s elapsed, ~2s remaining]
  Month 100/149  [4s elapsed, ~2s remaining]
  Month 120/149  [4s elapsed, ~1s remaining]
  Month 140/149  [5s elapsed, ~0s remaining]
  Done in 5.5s

  Mean RMSE by evaluation window:
    2 weeks     : 0.033268
    1 week      : 0.032827
    1 day       : 0.032200

DIAGNOSTIC: NaN entries in RMSE over surviving-eligible factors
  rmse[10] (2 weeks): 0 NaN entries (0.0000% of 149 months x 573 factors)
  rmse[5] (1 week): 0 NaN entries (0.0000% of 149 months x 573 factors)
  rmse[1] (1 day): 0 NaN entries (0.0000% of 149 months x 573 factors)

  If all three counts are 0 (or negligibly small), the per-window
  fill computed in Part E is a no-op in practice for this dataset --
  implemented anyway for correctness, not skipped.
80% Noise Filter (based on 1-week RMSE):
  Features with valid params:  573
  Features wi

C:\Users\Henry\AppData\Local\Temp\ipykernel_36112\1098613730.py:269: RuntimeWarning: Mean of empty slice
  avg_rmse_full = np.nanmean(rmse[5], axis=0)


In [1]:
"""
Autocorrelation truncation diagnostic, run explicitly rather than inline in
notebook 02's Part F, so the full lag trail is visible rather than only the
first three printed values and a final n_eff.

Answers three questions directly:
  1. How many lags were actually summed before truncation?
  2. At which lag did the |rho| < 0.05 stopping rule fire, and what was that
     lag's value (the one EXCLUDED from the sum)?
  3. Does the truncation lag / n_eff differ between agg_means and
     agg_full_moments? It shouldn't in principle, since monthly_fwd_63d is
     the same target series in both, but this checks rather than assumes it,
     since the two datasets may have a different n_months (agg_means and
     agg_full_moments can carry different numbers of valid rows depending on
     surviving-factor filtering upstream).

Run from the same directory as the Polymodel intermediate .npz files.
"""

import numpy as np
from pathlib import Path

DATA_PM = Path("../../Data/Results/Reproducing_Hellinger_Polymodel/intermediate")
DATASETS = ["agg_full_moments", "agg_means"]

THRESHOLD = 0.05
MAX_LAG = 30  # generous ceiling; the loop breaks well before this in practice


def truncated_sum(fwd_valid, threshold=THRESHOLD, max_lag=MAX_LAG):
    """
    Walks lag = 1, 2, 3, ... and sums rho_k until the first lag whose |rho_k|
    falls below `threshold`. That lag's value is NOT added to the sum -- the
    sum runs over lags 1..(stop_lag - 1) only.

    Returns a dict with the full per-lag trail plus the summary quantities,
    so the trail can be printed and the summary quantities used downstream
    without recomputing anything.
    """
    trail = []
    running_sum = 0.0
    stop_lag = None

    for lag in range(1, max_lag + 1):
        if lag >= len(fwd_valid):
            break
        rho = float(np.corrcoef(fwd_valid[:-lag], fwd_valid[lag:])[0, 1])
        below = abs(rho) < threshold
        trail.append({"lag": lag, "rho": rho, "below_threshold": below})
        if below:
            stop_lag = lag
            break
        running_sum += rho

    n_lags_summed = stop_lag - 1 if stop_lag is not None else len(trail)
    vif = 1 + 2 * running_sum
    n = len(fwd_valid)
    n_eff = n / vif if vif > 0 else np.nan

    return {
        "trail": trail,
        "stop_lag": stop_lag,
        "n_lags_summed": n_lags_summed,
        "sum_rho": running_sum,
        "vif": vif,
        "n_nominal": n,
        "n_eff": n_eff,
    }


def report(dataset_name):
    path = DATA_PM / f"hellinger_monthly_{dataset_name}.npz"
    if not path.exists():
        print(f"  !! {path} not found, skipping {dataset_name}")
        return None

    hell = np.load(path, allow_pickle=True)
    fwd = hell["monthly_fwd_63d"]
    fwd_valid = fwd[~np.isnan(fwd)]

    result = truncated_sum(fwd_valid)

    print("=" * 78)
    print(f"DATASET: {dataset_name}")
    print("=" * 78)
    print(f"  n (valid monthly forward returns): {result['n_nominal']}")
    print(f"  threshold for truncation: |rho| < {THRESHOLD}")
    print()
    print(f"  {'lag':>4} {'rho':>9} {'included?':>10}")
    print(f"  {'-' * 26}")
    for row in result["trail"]:
        tag = "STOP (excluded)" if row["below_threshold"] else "included"
        print(f"  {row['lag']:>4} {row['rho']:>+9.4f} {tag:>16}")

    if result["stop_lag"] is None:
        print(f"\n  !! loop reached max_lag={MAX_LAG} without finding a lag "
              f"below threshold -- extend MAX_LAG and re-run.")
    else:
        print(f"\n  Truncated at lag {result['stop_lag']} "
              f"(rho={result['trail'][-1]['rho']:+.4f}, "
              f"first |rho| < {THRESHOLD})")
    print(f"  Lags actually summed: 1..{result['n_lags_summed']} "
          f"({result['n_lags_summed']} lag(s) contributed to the VIF)")
    print(f"  Sum of included rho values: {result['sum_rho']:+.4f}")
    print(f"  Variance inflation factor (1 + 2*sum): {result['vif']:.3f}")
    print(f"  n_eff = n / VIF = {result['n_nominal']} / {result['vif']:.3f} "
          f"= {result['n_eff']:.1f}")
    print()

    return result


if __name__ == "__main__":
    results = {}
    for ds in DATASETS:
        results[ds] = report(ds)

    valid = {k: v for k, v in results.items() if v is not None}
    if len(valid) == 2:
        print("=" * 78)
        print("CROSS-DATASET COMPARISON")
        print("=" * 78)
        keys = list(valid.keys())
        a, b = valid[keys[0]], valid[keys[1]]
        print(f"  {'':30} {keys[0]:>18} {keys[1]:>18}")
        print(f"  {'n (nominal)':30} {a['n_nominal']:>18} {b['n_nominal']:>18}")
        print(f"  {'stop lag':30} {a['stop_lag']:>18} {b['stop_lag']:>18}")
        print(f"  {'lags summed':30} {a['n_lags_summed']:>18} {b['n_lags_summed']:>18}")
        print(f"  {'sum of rho':30} {a['sum_rho']:>+18.4f} {b['sum_rho']:>+18.4f}")
        print(f"  {'VIF':30} {a['vif']:>18.3f} {b['vif']:>18.3f}")
        print(f"  {'n_eff':30} {a['n_eff']:>18.1f} {b['n_eff']:>18.1f}")
        same_target = np.array_equal(
            np.load(DATA_PM / f"hellinger_monthly_{keys[0]}.npz", allow_pickle=True)["monthly_fwd_63d"],
            np.load(DATA_PM / f"hellinger_monthly_{keys[1]}.npz", allow_pickle=True)["monthly_fwd_63d"],
            equal_nan=True,
        )
        print(f"\n  monthly_fwd_63d identical between datasets: {same_target}")
        if same_target:
            print("  (expected: both datasets predict the same S&P 100 target;")
            print("   any difference in n_nominal reflects filtering upstream,")
            print("   not a different target series)")

DATASET: agg_full_moments
  n (valid monthly forward returns): 146
  threshold for truncation: |rho| < 0.05

   lag       rho  included?
  --------------------------
     1   +0.4852         included
     2   +0.1577         included
     3   -0.1809         included
     4   -0.0562         included
     5   +0.0645         included
     6   +0.1610         included
     7   +0.2488         included
     8   +0.1352         included
     9   +0.0284  STOP (excluded)

  Truncated at lag 9 (rho=+0.0284, first |rho| < 0.05)
  Lags actually summed: 1..8 (8 lag(s) contributed to the VIF)
  Sum of included rho values: +1.0152
  Variance inflation factor (1 + 2*sum): 3.030
  n_eff = n / VIF = 146 / 3.030 = 48.2

DATASET: agg_means
  n (valid monthly forward returns): 146
  threshold for truncation: |rho| < 0.05

   lag       rho  included?
  --------------------------
     1   +0.4852         included
     2   +0.1577         included
     3   -0.1809         included
     4   -0.0562       